In [ ]:
# 1

# Tiandra Taylor

In [2]:
# 2

# imports
import pandas as pd
import os 

path = "C:\\Users\\Tiand\\Dropbox\\5330corgis"

for i in os.listdir(path):
    print(i)

.ipynb_checkpoints
102119-8469.csv
102120-8243.csv
102121-5390.csv
102121-7380.csv
102122-4015.csv
102123-3428.csv
102124-3988.csv
102124-9377.csv
desktop.ini
i3_ ClayWillden.ipynb
newcorgis-20220618.csv
newcorgis-20220619.csv
newreg-062822.csv
pdx623.csv
portlandeventjun22.csv
signupsjune272022.csv
tacoma-062322.csv


In [75]:
# 3

# pre-works
import psycopg2 as psy

issueslist = []

rownum = 0

# connect to db 
conn = psy.connect(
    database='corgirace',
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    host=os.getenv('DB_HOST'),
    port='5432'
)

cursor = conn.cursor()

# loop through list 
for i in os.listdir(path):
    # non-csv? - skip
    # csv?? then ->
    if i[-3:] == 'csv':
        
        # ingested or not?
        qrychck = """
        SELECT COUNT(*) as added_or_not
        FROM ingest
        WHERE filename = '%s'
        """ % (i)
        
        cursor.execute(qrychck)

        # use fetchone to get row 
        y = cursor.fetchone()
        
        # if else statement 
        if y[0] > 0:
            print("\n", "File:",i,"has already been processed.")
        else:
            print("\n", i,"Not ingested yet.")
            # read in csv
            py_obj = open(path+'\\'+i, "r")
            # skip header row 
            rownum = 0
            for x in py_obj:
                # not header
                if (rownum > 0):
                    # strip
                    x = x.strip()
                    # split
                    x = x.split(",")
                    # issueslist be blank
                    issueslist = []
                    # num of col in line? 
                    # more than 5? -> too many
                    if len(x) > 5:
                        issueslist.append("Too many Columns")
                    # less than 5? -> not enough
                    elif len(x) < 5:
                        issueslist.append("Too Few Columns")
                    # length of val in name col 
                    # less than 1? -> too short
                    elif len(x[0]) < 1:
                        issueslist.append("Name Too Short")
                    # more than 100? -> too long
                    elif len(x[0]) > 100:
                        issueslist.append("Name Too Long")
                    # first character in breed P or C? -> learned you could do to index instead of doing two for loops to get first letter or second item in list!! shoutout to claude for that!
                    # must check if it is empty or we get errors
                    # don't use or use not in otherwise it will always return true
                    elif len(x[1]) == 0 or (x[1][0]).upper() not in ['P', 'C']:
                        issueslist.append("Breed Issue")
                    # Gender col (F, M, SF, NM)
                    elif len(x[2]) == 0 or x[2].lower() not in ['m', 'f', 'sf', 'nm']:
                        issueslist.append("Gender Col Issue")
                    # Weight -> must be numerical
                    try:
                        weight = float(x[3])
                        # below 15 then weight too low
                        if weight < 15:
                            issueslist.append("Weight Too Low")
                        # above 35 then weight too high
                        if weight > 35:
                            issueslist.append("Weight Too High")
                    except:
                        issueslist.append("Weight Non-Numeric")
                    # age = numreic val
                    try:
                        age = float(x[4])
                        # age < 1 too young
                        if age < 1:
                            issueslist.append("Age Too Low")
                        if age > 10:
                            issueslist.append("Age Too High")
                        # age > 10 too old
                    except:
                        issueslist.append("Age Non-Numeric")
                    # to console -> name o ffile, value of name col, issues list,
                    # use a f string because commas make an ugly output lol
                    print(f"File: {i} {x[0]} {issueslist}")
                else:
                    rownum += 1



 102119-8469.csv Not ingested yet.
File: 102119-8469.csv Neck-Nipping Nellie ['Gender Col Issue', 'Age Too High']
File: 102119-8469.csv Finnleberry Huck []
File: 102119-8469.csv Wyvern []
File: 102119-8469.csv Ogre []
File: 102119-8469.csv Zap []
File: 102119-8469.csv  ['Name Too Short']

 102120-8243.csv Not ingested yet.
File: 102120-8243.csv Megawoof ['Weight Too High', 'Age Too High']
File: 102120-8243.csv Charles ['Gender Col Issue', 'Weight Too High']
File: 102120-8243.csv Mabel the Claw []
File: 102120-8243.csv Instant Breakfast []
File: 102120-8243.csv Willow The Dust Storm Corgifriend []
File: 102120-8243.csv Empanada []
File: 102120-8243.csv Leatherface []
File: 102120-8243.csv Corgi Elliot []
File: 102120-8243.csv Tobias []
File: 102120-8243.csv Nature Girl Nala ['Gender Col Issue', 'Weight Non-Numeric']
File: 102120-8243.csv Mountain Man []
File: 102120-8243.csv Loofah []
File: 102120-8243.csv Amelia the Woof ['Breed Issue']

 102121-5390.csv Not ingested yet.
File: 102121

In [19]:
# 4

# had to connect to corgi db to check if we had ingest the files or not in question 3. 
corg_excep = '''
CREATE TABLE IF NOT EXISTS corgi_exception (
exceptid SERIAL PRIMARY KEY,
except_record TEXT,
origin_file VARCHAR(100) NOT NULL,
issues TEXT,
except_timestamp TIMESTAMP,
fixed_timestamp TIMESTAMP
);
'''

cursor.execute(corg_excep)

conn.commit()

print("Created corgi_exception Table")

Created corgi_exception Table


In [81]:
# 5

# 3

# pre-works
import psycopg2 as psy

issueslist = []

rownum = 0

# loop through list 
for i in os.listdir(path):
    # non-csv? - skip
    # csv?? then ->
    if i[-3:] == 'csv':
        
        # ingested or not?
        qrychck = """
        SELECT COUNT(*) as added_or_not
        FROM ingest
        WHERE filename = '%s'
        """ % (i)
        
        cursor.execute(qrychck)

        # use fetchone to get row 
        y = cursor.fetchone()
        
        # if else statement 
        if y[0] > 0:
            print("\n", "File:",i,"has already been processed.")
        else:
            print("\n", i,"Not ingested yet.")
            # read in csv
            py_obj = open(path+'\\'+i, "r")
            # skip header row 
            rownum = 0
            for x in py_obj:
                # not header
                if (rownum > 0):
                    # strip
                    x = x.strip()
                    # split
                    x = x.split(",")
                    # issueslist be blank
                    issueslist = []
                    # num of col in line? 
                    # more than 5? -> too many
                    if len(x) > 5:
                        issueslist.append("Too many Columns")
                    # less than 5? -> not enough
                    elif len(x) < 5:
                        issueslist.append("Too Few Columns")
                    # length of val in name col 
                    # less than 1? -> too short
                    elif len(x[0]) < 1:
                        issueslist.append("Name Too Short")
                    # more than 100? -> too long
                    elif len(x[0]) > 100:
                        issueslist.append("Name Too Long")
                    # first character in breed P or C? -> learned you could do to index instead of doing two for loops to get first letter or second item in list!! shoutout to claude for that!
                    # must check if it is empty or we get errors
                    # don't use or use not in otherwise it will always return true
                    elif len(x[1]) == 0 or (x[1][0]).upper() not in ['P', 'C']:
                        issueslist.append("Breed Issue")
                    # Gender col (F, M, SF, NM)
                    elif len(x[2]) == 0 or x[2].lower() not in ['m', 'f', 'sf', 'nm']:
                        issueslist.append("Gender Col Issue")
                    # Weight -> must be numerical
                    try:
                        weight = float(x[3])
                        # below 15 then weight too low
                        if weight < 15:
                            issueslist.append("Weight Too Low")
                        # above 35 then weight too high
                        if weight > 35:
                            issueslist.append("Weight Too High")
                    except:
                        issueslist.append("Weight Non-Numeric")
                    # age = numreic val
                    try:
                        age = float(x[4])
                        # age < 1 too young
                        if age < 1:
                            issueslist.append("Age Too Low")
                        if age > 10:
                            issueslist.append("Age Too High")
                        # age > 10 too old
                    except:
                        issueslist.append("Age Non-Numeric")
                    # to console -> name o ffile, value of name col, issues list,
                    # use a f string because commas make an ugly output lol
                    print(f"File: {i} {x[0]} {issueslist}")

                    # insert into good db
                    if len(issueslist) == 0:
                        # take care of ' 
                        if "'" in x[0]:
                            x[0] = x[0].replace("'", '')
                        # Truncate name to 30 characters
                        x[0] = x[0][:30]
                        # Pem or Cardi
                        if (x[1][0]).upper() == 'P':
                            x[1] = 'Pem'
                        if (x[1][0]).upper() == 'C':
                            x[1] = 'Cardi'
                        # Gender
                        if x[2].upper() == 'M':
                            x[2] = 'M'
                        if x[2].upper() == 'F':
                            x[2] = 'F'
                        if x[2].upper() == 'NM':
                            x[2] = 'NM'
                        if x[2].upper() == 'SF':
                            x[2] = 'SF'
                        # weight
                        x[3] = float(x[3])
                        x[3] = round(x[3])
                        x[3] = int(x[3])
                        # age
                        x[4] = float(x[4])
                        x[4] = round((x[4] * 2) / 2)
                        x[4] = int(x[4])

                    
                        # insert statement - might be wrong?
                        cursor.execute(''' INSERT INTO corgi (corgname, breed, gender, weight, age, fromfile)
                        VALUES ('%s', '%s', '%s', %d, %d, '%s')''' % (x[0], x[1], x[2], x[3], x[4], i))

                        conn.commit()

                        print(f"Record created for {x[0]}.")
                    else:
                        # handle non-compliant records
                        # create except_record with | delimiter, remove single quotes
                        except_record = ''
                        for w in range(len(x)):
                            if w == 0:
                                except_record = x[w]
                            else:
                                except_record = except_record + '|' + x[w]
                        
                        # remove single guys
                        except_record = except_record.replace("'", '')
                        
                        # create issues string with | delimiter
                        issues_string = ''
                        for q in range(len(issueslist)):
                            if q == 0:
                                issues_string = issueslist[q]
                            else:
                                issues_string = issues_string + '|' + issueslist[q]
                        
                        # corgi_exception table insert 
                        cursor.execute(''' INSERT INTO corgi_exception (except_record, origin_file, issues, except_timestamp)
                        VALUES ('%s', '%s', '%s', current_timestamp)''' % (except_record, i, issues_string))
                        
                        conn.commit()
                    
                        print(f"Record for {x[0]} had these issues: {issueslist}.")
                else:
                    rownum += 1
                    # purgatory
            # ingest table insert 
            cursor.execute(''' INSERT INTO ingest (filename, whendone)
            VALUES ('%s', current_timestamp)''' % (i))
    
            conn.commit()
    
            print(f"Ingestion complete for {i}.")


 102119-8469.csv Not ingested yet.
File: 102119-8469.csv Neck-Nipping Nellie ['Gender Col Issue', 'Age Too High']
Record for Neck-Nipping Nellie had these issues: ['Gender Col Issue', 'Age Too High'].
File: 102119-8469.csv Finnleberry Huck []
Record created for Finnleberry Huck.
File: 102119-8469.csv Wyvern []
Record created for Wyvern.
File: 102119-8469.csv Ogre []
Record created for Ogre.
File: 102119-8469.csv Zap []
Record created for Zap.
File: 102119-8469.csv  ['Name Too Short']
Record for  had these issues: ['Name Too Short'].
Ingestion complete for 102119-8469.csv.

 102120-8243.csv Not ingested yet.
File: 102120-8243.csv Megawoof ['Weight Too High', 'Age Too High']
Record for Megawoof had these issues: ['Weight Too High', 'Age Too High'].
File: 102120-8243.csv Charles ['Gender Col Issue', 'Weight Too High']
Record for Charles had these issues: ['Gender Col Issue', 'Weight Too High'].
File: 102120-8243.csv Mabel the Claw []
Record created for Mabel the Claw.
File: 102120-8243.c

In [82]:
# 6

corgi_excep = '''SELECT * FROM corgi_exception ORDER BY except_record LIMIT 10;'''

cursor.execute(corgi_excep)

df = pd.read_sql(corgi_excep, conn, index_col='exceptid')
print(df)

                            except_record      origin_file  \
exceptid                                                     
24                 Amelia the Woof|||27|3  102120-8243.csv   
38               Baby Congrats|PWC|M|36|4  102123-3428.csv   
28        Bailey Barkbark|Cardi|SF|26|2||  102121-7380.csv   
27                    Bandit|Pem|M|31|4||  102121-7380.csv   
33                   Bowser|Pem|NM|27|4||  102121-7380.csv   
39                       |Cardi|SF|26|4.5  102123-3428.csv   
22                    Charles|Pem||36|2.5  102120-8243.csv   
40                Crossbow Jo|Pem|R|5|2.7  102124-3988.csv   
52         Double-Dragon|Cardi|NM|32|3.5|  102124-9377.csv   
36          Double-Plus|Pem|SF|Heavy!|4.5  102122-4015.csv   

                                    issues           except_timestamp  \
exceptid                                                                
24                             Breed Issue 2025-10-25 04:53:59.179558   
38                         Weight To

C:\Users\Tiand\AppData\Local\Temp\ipykernel_7964\1170916701.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(corgi_excep, conn, index_col='exceptid')


In [109]:
# 7

corg = '''SELECT c.corgid, c.corgname, c.weight, c.age, c.fromfile FROM CORGI as c JOIN ingest as i on c.fromfile = i.filename WHERE DATE(i.whendone) = CURRENT_DATE ORDER BY corgname LIMIT 10;'''
#corg = '''SELECT * FROM corgi'''
cursor.execute(corg)

df = pd.read_sql(corg, conn, index_col='corgid')
print(df)

                corgname  weight  age         fromfile
corgid                                                
148            Baby Bear      29  3.0  102122-4015.csv
152         Brunchkiller      27  4.0  102123-3428.csv
155            Campstove      22  2.0  102123-3428.csv
139         Corgi Elliot      24  3.0  102120-8243.csv
149             Corgihor      28  4.0  102122-4015.csv
156      Delbert the Cog      31  4.0  102124-3988.csv
151         Demi Locorgo      25  4.0  102122-4015.csv
144            Drumstick      23  2.0  102121-5390.csv
137             Empanada      26  2.0  102120-8243.csv
130     Finnleberry Huck      26  3.0  102119-8469.csv


C:\Users\Tiand\AppData\Local\Temp\ipykernel_7964\3665637263.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(corg, conn, index_col='corgid')
